# Tutorial · Day 3 企业知识图谱 + GraphRAG (v6.0)

## 牛津 Tutorial 仿真 · Persona

> **You are an Oxford tutorial fellow in Knowledge Graphs and GraphRAG.** Your tutee is a mid-career data scientist building an enterprise marketing KG.
>
> **Persona rules (strict):**
> 1. **Never give direct answers.** Do not write code for the student. Do not say "the answer is X." If asked "is X correct?", reply with a counter-question. 不直接给答案, 不直接答.
> 2. **Use Socratic questioning.** Every turn ends with a probing question (why / how could / what if / counterexample / 凭什么 / 依据 / 若…变). 苏格拉底式追问.
> 3. **Reject vague claims.** If the student says "GraphRAG is better," demand: "Better at what task? By what metric? Compared to what baseline? 凭什么?"
> 4. **Devil's advocate (Christensen Center HBS).** Take the opposite stance whenever the student asserts--force them to defend.
> 5. **Scaffold fade.** If defense fails twice, drop one scaffold level (hint -> partial code skeleton -> pointer to notes.md §关键回顾), but still no direct answer.
> 6. **限频 (防依赖):** 1 次/天 per unit (Vygotsky 共构边界). 第 2 次同日访问 -> 引导回 schedule.json 复习 + 退出。
> 7. **Exit artifact:** End session with 2-3 盲点 + 推荐复习单元 + 写入 student_model.json。
>
> **Topic**: Enterprise Knowledge Graph (EKG) · KGE (TransE/RotatE/ComplEx) · GraphRAG (微软2024 arXiv 2404.16130, Global/Local/DRIFT search) · networkx / Neo4j / LangChain GraphRAG / langchain_experimental.LLMGraphTransformer.


## Pre-tutorial Task (强制 retrieval · 提交后 tutorial 才开始)

**Before this tutorial, you must submit (in <300 words):**

1. 用 `networkx.MultiDiGraph` 构建一个含 ≥7 节点 ≥8 边的营销知识图谱的本体设计（列出实体类与关系类）
2. 写出 TransE 的得分函数 f(h,r,t) 与 margin loss L，并解释为何 TransE 无法建模"相似"这类对称关系
3. 用一句话回答：GraphRAG 的 DRIFT Search 为何比纯 Local Search 更适合回答"竞品 A 和 B 的共同弱点是什么"？

> **不提交不能进 tutorial**--这是 retrieval practice（Butler 2010：检索练习 68% vs 重学 44%）。提交后写入下方 `student_pre_task`。


In [ ]:
# === Socratic Multi-turn Loop (静态 if/else 仿真, 不调 LLM API) ===
# 设计依据: Oxford tutorial 1对1-3 + arxiv 2409.05511 / 2507.05795 / 2508.21204 Socratic LLM 论文
# 每轮检测 defense 失败则降一级 scaffold, 仍禁直接答案

import json

student_pre_task = {
    "ontology": "产品-品牌-品类-客户-评论-活动-渠道 7 类节点；买了/属于/互补/评论了/参加了/分发于 6 类边",
    "transe": "f=-||h+r-t||; L=Σ max(0, γ+f(h,t)-f(h',t')); TransE 一对一平移无法处理对称关系",
    "drift": "DRIFT 先全局定位再局部精确检索, 比 Local Search 多了全局视角"
}

# 5 轮 Socratic 追问 (静态分支, 仿真 LLM Socratic)
tutorial_log = []
scaffold_level = 0  # 0=裸问, 1=给hint, 2=指向notes.md

def socratic_turn(round_id, student_answer, probe_topic):
    """单轮 Socratic: 检测 defense 质量, 失败则降 scaffold, 仍禁直接答案."""
    global scaffold_level
    turn = {"round": round_id, "topic": probe_topic, "student_answer": student_answer}

    # --- Round 1: 本体设计辩护 (ILO1) ---
    if round_id == 1:
        if "活动" in student_answer and "渠道" in student_answer and len(student_answer) > 40:
            turn["tutor_reply"] = ("你列了 7 类实体, 但凭什么'活动'和'渠道'是必要的? "
                                   "若你的业务是纯 DTC 电商, 没有渠道分销, 这两类节点会带来什么冗余? "
                                   "反例: 给我一个场景, 删掉'渠道'反而让多跳推理更准.")
            turn["socratic_questions"] = [
                "why '活动' 与 '渠道' 是必要的而非可删的?",
                "若业务是纯 DTC 电商, 这两类节点是否冗余?",
                "反例: 什么场景下删掉'渠道'反而让多跳推理更准?"
            ]
        else:
            turn["tutor_reply"] = ("你的本体里少了至少 2 类实体. 为什么? "
                                   "客户买了产品后会留下什么? 产品由谁制造? 通过什么渠道分发? "
                                   "不直接告诉你缺什么--回去看 notes.md 营销映射表, 再回来答.")
            turn["socratic_questions"] = [
                "客户买了产品后会留下什么?",
                "产品由谁制造?",
                "通过什么渠道分发?",
                "凭什么你的 5 类实体够用?"
            ]
            scaffold_level = max(scaffold_level, 1)

    # --- Round 2: networkx API 追问 (ILO1) ---
    elif round_id == 2:
        if "successors" in student_answer or "predecessors" in student_answer:
            turn["tutor_reply"] = ("你用了 successors--好. 但 MultiDiGraph 和 DiGraph 的区别是什么? "
                                   "若同两个节点之间有 '买了' 和 '评论了' 两条边, 你的查询会返回什么? "
                                   "如何区分? 反例: 给我一个查询, 用 neighbors 会出错但 successors 不会.")
            turn["socratic_questions"] = [
                "MultiDiGraph 和 DiGraph 的区别是什么?",
                "两节点间有'买了'和'评论了'两条边, 你的查询返回什么?",
                "如何区分这两条边?",
                "反例: 什么查询用 neighbors 出错但 successors 不会?"
            ]
        else:
            turn["tutor_reply"] = ("你说用 neighbors--有向图能用 neighbors 吗? "
                                   "为什么 networkx 给你 success/predecessors 两套 API? "
                                   "Neo4j 的 Cypher `MATCH (n)-[r]->(m)` 里这个 `->` 在暗示什么? "
                                   "回去查 MultiDiGraph 文档, 再答.")
            turn["socratic_questions"] = [
                "有向图能用 neighbors 吗?",
                "为什么 networkx 给 successors/predecessors 两套?",
                "Neo4j Cypher 的 `->` 暗示什么?",
                "若用 neighbors 会漏掉什么?"
            ]
            scaffold_level = max(scaffold_level, 1)

    # --- Round 3: TransE 对称关系辩护 (ILO2) ---
    elif round_id == 3:
        if "一对多" in student_answer or "对称" in student_answer or "h+r=t" in student_answer.replace(" ", ""):
            turn["tutor_reply"] = ("你说 TransE 无法建模对称关系--凭什么? "
                                   "若'相似'是对称的, 即 sim(A,B)=sim(B,A), TransE 要求 A+r≈B 且 B+r≈A, "
                                   "这两式同时成立意味着什么? 数学上推出矛盾. "
                                   "那 RotatE 凭什么能解决? 复数旋转的什么性质让它能处理对称? "
                                   "反例: ComplEx 又凭什么?")
            turn["socratic_questions"] = [
                "凭什么 TransE 无法建模对称关系?",
                "A+r≈B 且 B+r≈A 同时成立意味着什么?",
                "数学上能推出矛盾吗?",
                "RotatE 复数旋转的什么性质解决对称?",
                "反例: ComplEx 又凭什么?"
            ]
        else:
            turn["tutor_reply"] = ("你的回答只有公式没有推理. 为什么 h+r≈t 这个'平移'模型天然反对称? "
                                   "画一张二维图: h=[0,0], r=[1,0], t=[1,0]. 若关系对称, r 应该等于什么? "
                                   "回去看 notes.md 关键回顾 2 的对比表, 再回来辩护.")
            turn["socratic_questions"] = [
                "为什么 h+r≈t 这个平移模型天然反对称?",
                "若关系对称, r 应该等于什么?",
                "画二维图: h=[0,0], r=[1,0], t=[1,0] 时若对称, 矛盾在哪?",
                "RotatE 凭什么解决?"
            ]
            scaffold_level = max(scaffold_level, 1)

    # --- Round 4: GraphRAG vs 传统 RAG 辩护 (ILO3) ---
    elif round_id == 4:
        if "多跳" in student_answer or "关系链" in student_answer:
            turn["tutor_reply"] = ("你说 GraphRAG 靠多跳关系链取胜--但构建 GraphRAG 需要 LLMGraphTransformer 抽取实体关系, "
                                   "成本远高于 TF-IDF. 凭什么值? "
                                   "给我一个反例: 什么场景下 GraphRAG 反而不如 TF-IDF? "
                                   "(提示: 简单事实问答.) "
                                   "那 DRIFT Search 又比 Local Search 多了什么? 凭什么贵?")
            turn["socratic_questions"] = [
                "GraphRAG 构建成本高于 TF-IDF, 凭什么值?",
                "反例: 什么场景下 GraphRAG 反而不如 TF-IDF?",
                "DRIFT Search 比 Local Search 多了什么?",
                "凭什么 DRIFT 比 Local 贵?",
                "若业务 80% 是简单事实问答, 你会推荐 GraphRAG 吗?"
            ]
        else:
            turn["tutor_reply"] = ("你说 GraphRAG 更好--更好是什么意思? 在什么任务上? 用什么指标? 对比什么基线? "
                                   "这个'更好'是召回率还是准确率? 是多跳问题还是单跳问题? "
                                   "回去看 notes.md 关键回顾 3 的对比表, 把'更好'具体到指标和场景, 再回来.")
            turn["socratic_questions"] = [
                "更好是什么意思?",
                "在什么任务上更好?",
                "用什么指标衡量?",
                "对比什么基线?",
                "多跳问题还是单跳问题?"
            ]
            scaffold_level = max(scaffold_level, 1)

    # --- Round 5: 反事实 + 元认知 (ILO4) ---
    elif round_id == 5:
        turn["tutor_reply"] = ("最后一个反事实: 假设微软 2024 没发 GraphRAG, 你会用什么方案回答'竞品 A 和 B 的共同弱点'? "
                               "纯 networkx 多跳检索 + LLM 摘要够吗? 凭什么? "
                               "再问: 你的企业客户预算只够选 GraphRAG 或 Neo4j 图数据库之一, 你选哪个? 为什么? "
                               "盲点自检: 你这次 tutorial 里答得最虚的是哪一轮?")
        turn["socratic_questions"] = [
            "假设没有 GraphRAG, 你会用什么方案回答共同弱点问题?",
            "纯 networkx 多跳 + LLM 摘要够吗? 凭什么?",
            "预算只够 GraphRAG 或 Neo4j 之一, 选哪个? 为什么?",
            "盲点自检: 你答得最虚的是哪一轮?",
            "若明天要给 CTO 汇报, 你最不敢讲的是哪部分?"
        ]

    tutorial_log.append(turn)
    return turn

# 跑 5 轮 Socratic (静态学生响应仿真)
mock_answers = [
    "我列了产品/品牌/品类/客户/评论 5 类节点, 买了/属于/评论了 3 类边",
    "用 G.neighbors(node) 获取邻居",
    "f=-||h+r-t||, L=Σ max(0,γ+f(h,t)-f(h',t')), TransE 一对一所以无法一对多",
    "GraphRAG 靠多跳关系链检索, 召回率比 TF-IDF 高",
    "没有 GraphRAG 我会用 networkx 多跳+LLM 摘要, 但社区摘要缺; 选 Neo4j 因生产稳定",
]

for i, ans in enumerate(mock_answers, 1):
    socratic_turn(i, ans, ["本体设计","networkx API","TransE 对称","GraphRAG 成本","反事实"][i-1])

print(f"=== Socratic Tutorial 完成, 共 {len(tutorial_log)} 轮 ===")
print(f"scaffold_level (0=裸问,1=hint,2=notes指针): {scaffold_level}")
print(f"总 Socratic 问题数: {sum(len(t['socratic_questions']) for t in tutorial_log)}")
for t in tutorial_log:
    print(f"\n--- Round {t['round']} ({t['topic']}) ---")
    print(f"Tutor: {t['tutor_reply'][:120]}...")


In [ ]:
# === student_model.json (跨单元复用) ===
# 记录掌握度/盲点/复习指针, 下次 tutorial 读入续讲

import json, os

student_model = {
    "student_id": "demo_student_v6",
    "unit": "day-3-knowledge-graph-graphrag",
    "last_updated": "2026-07-25",
    "mastery": {
        "ILO1_KG_build": {"level": "partial", "attempts": 1, "last_score": 0.4,
                          "blind_spots": ["遗漏'活动'和'渠道'两类实体", "未论证本体必要性"]},
        "ILO2_TransE": {"level": "partial", "attempts": 1, "last_score": 0.6,
                        "blind_spots": ["TransE 对称关系矛盾的数学推导不完整", "RotatE 复数旋转解决对称的机制未答"]},
        "ILO3_GraphRAG": {"level": "partial", "attempts": 1, "last_score": 0.6,
                          "blind_spots": ["未给 GraphRAG 不如 TF-IDF 的反例", "DRIFT vs Local 成本差异未量化"]},
        "ILO4_migration": {"level": "not_assessed", "attempts": 0, "last_score": None,
                           "blind_spots": []}
    },
    "scaffold_level": scaffold_level,
    "weak_loop_triggered": scaffold_level >= 1,
    "recommended_review": [
        "notes.md 关键回顾 1 (本体设计)",
        "notes.md 关键回顾 2 (TransE/RotatE/ComplEx 对比表)",
        "schedule.json C1 (TransE loss) + C3 (KGE 三方法)",
        "practice.md D1 Worked 阶段重看",
        "reading.md TransE / RotatE 条目"
    ],
    "next_unit_pointer": "day-4-multimodal-fusion (KGE margin-based 对比学习与 CLIP/BLIP 共享数学直觉)",
    "daily_usage_count": 1,
    "daily_limit": 1
}

student_model_path = "student_model.json"
with open(student_model_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

print(f"student_model.json 已写入: {student_model_path}")
print(f"掌握度摘要:")
for k, v in student_model["mastery"].items():
    print(f"  {k}: {v['level']} (score={v['last_score']}, blind_spots={len(v['blind_spots'])})")
print(f"\n弱项循环触发: {student_model['weak_loop_triggered']}")
print(f"推荐复习单元数: {len(student_model['recommended_review'])}")


## Hattie (2007) 四级 Formative Feedback

> Hattie, J. (2007). *The Power of Feedback*. Review of Educational Research, 77(1), 81-112.
> 4 级: TASK / PROCESS / SELF-REG / FEED-FORWARD. **避免 Self 级表扬**（Hattie 实证: 表扬效应量仅 0.14, 远低于 TASK 0.75 / PROCESS 0.65）。

基于本次 tutorial 5 轮 Socratic 的学生表现, 给出四级反馈:

### [TASK] 任务级 (针对本次 tutorial 答案的对错与质量)
- **R1 本体设计**: 遗漏"活动"与"渠道"两类实体 -- 不通过. 重做: 回到 notes.md 营销映射表核对 7 类实体, 至少补 2 类.
- **R2 networkx API**: 错用 `neighbors` -- 不通过. MultiDiGraph 有向, 必须用 `successors`/`predecessors`. Neo4j Cypher `MATCH (n)-[r]->(m)` 的 `->` 即有向.
- **R3 TransE 对称**: 公式对但推理不全 -- 部分通过. 补数学推导: A+r≈B 且 B+r≈A => 2r≈0 => r≈0, 退化为无效关系, 故 TransE 无法建模对称.
- **R4 GraphRAG 成本**: 部分通过. 未给反例(简单事实问答 GraphRAG 不如 TF-IDF). 补一个具体反例.
- **R5 反事实**: 通过. 知道无 GraphRAG 时用 networkx 多跳+LLM 摘要, 知道预算约束下选 Neo4j.

### [PROCESS] 过程级 (针对学生的策略与方法)
- 你的策略问题: 每轮回答都先抛结论再(被追问才)补理由 -- 反了. **先推理再下结论**, 用"因为...所以..."结构.
- 检索习惯: 多次说"凭直觉"--学习科学称此为"幻觉性流畅感"(illusion of fluency). **每次结论必须落到公式/库 API/数字**.
- 反例思维不足: 4/5 轮被追问反例才补. 主动构造反例是 devil's advocate 的核心能力.

### [SELF-REG] 自我调节级 (针对学生的元认知与监控)
- 你在第 3 轮(TransE 对称)答得最虚 -- 这就是你的盲点信号. 下次遇到"无法用数学推导"时, 主动标记"此处我不确定", 而非用模糊词掩盖.
- 盲点自检: 在 student_model.json 里你回答最虚的是 R3, 与我观察一致 -- 你的元认知是准的, 但未提前声明.
- 下次 tutorial 前: 先做 1 次 self-explanation(对自己讲一遍 TransE 对称矛盾的推导), 录音回放找卡壳点.

### [FEED-FORWARD] 前馈级 (针对下一单元与长期路径)
- **下一步**: 触发 weak_loop (scaffold_level=1). 回到 practice.md D1 Worked 阶段重看, 然后做 D1 Faded.
- **复习指针**: schedule.json C1 (TransE loss) + C3 (KGE 三方法对比) 21 天内不漏复习.
- **跨单元**: Day 4 多模态融合的 CLIP/BLIP 对比学习与本单元 TransE margin loss 共享"正负样本对比"数学直觉 -- 复习时主动连接.
- **进阶路径**: 若想深入, 读 pykeen 库源码的 TransE/RotatE 实现, 对比自己 numpy 版本的差异.


## 限频与 Exit

### 限频 (防依赖 · Vygotsky 共构边界)
- **每单元每天 1 次 tutorial** (1次/天). 今日已用 1 次.
- 第 2 次同日访问 -> 引导回 `schedule.json` 做间隔复习 + 退出. 不再开新 tutorial.
- 依据: Oxford tutorial 每周 1 次, 强制学生独立消化. LLM 仿真无限频会导致"问依赖"--学生不再主动检索.

### Exit Artifact (退出产物)
本次 tutorial 结束前, 你必须:
1. **列出 2-3 个盲点** (基于 student_model.json 的 blind_spots):
   - 盲点 1: TransE 对称关系矛盾的数学推导 (A+r≈B 且 B+r≈A => 2r≈0)
   - 盲点 2: networkx MultiDiGraph 有向 API (successors vs neighbors)
   - 盲点 3: GraphRAG vs TF-IDF 的反例场景 (简单事实问答)
2. **推荐复习单元**:
   - `practice.md` D1 Worked 阶段重看 + D1 Filled
   - `schedule.json` C1 + C3 (今天复习, 3 天后, 8 天后, 21 天后, 60 天后, 180 天后)
   - `notes.md` 关键回顾 1-3
   - `alignment.md` ILO1-ILO3 的 TLA 列
3. **下次 tutorial 前置任务**:
   - 完成 practice.md D1 Faded 阶段 (3 reps)
   - 重写 pre-task 的 3 道 (尤其本体设计要补活动/渠道)
   - 手算 1 轮 TransE 梯度更新 (h=[0,0], r=[1,0], t=[1,0], γ=1, lr=0.1, t'=[2,0])

### 退出确认
- [ ] student_model.json 已更新
- [ ] 2-3 盲点已列出
- [ ] schedule.json 复习卡片已 due
- [ ] 下次前置任务已记录

**Session 结束. 明日再见.**

---

*tutorial.ipynb 依据 Oxford tutorial (1对1-3, 每周, 口头辩护) + arxiv 2024-2025 Socratic LLM 论文 (2409.05511/2507.05795/2508.21204) + Hattie (2007) 4 级 feedback 设计. LLM 仿真为静态 if/else, 不调 API.*
